<a href="https://colab.research.google.com/github/sprince0031/ICT-Python-ML/blob/main/Neural%20Computing/DQN_classic_control.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning for Classic Control (CartPole-v1)

**Based on Hands-on Machine Learning, Chapter 18: Reinforcement Learning**

In this session, we implement a DQN agent from scratch using TensorFlow and Keras.

## 1. The Environment & Setup

In [ ]:
!pip install gymnasium


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import gymnasium as gym
import random

if not tf.config.list_physical_devices('GPU'):
    print("No GPU found.")

# Create the environment
env = gym.make("CartPole-v1", render_mode="rgb_array")

# Reset to get the first observation
obs, info = env.reset(seed=42)

#Visualization function
def plot_environment(env, figsize=(5,4)):
    plt.figure(figsize=figsize)
    img = env.render()
    plt.imshow(img)
    plt.axis("off")
    return img

plot_environment(env)
plt.show()


### obs

Each observation is a 1D NumPy array containing four floats:
* cart's horizontal position (0.0 = center)
* cart's velocity (positive means right)
* the angle of the pole (0.0 = vertical)
* angular velocity of the pole (positive means clockwise)

In [ ]:
obs

### info

Helps debugging

In [ ]:
info

## 2. The Q-Network

We approximate the Q-Value function Q(s,a) using a neural network. The input is the state, and the output is the Q-value for each action.

**Neural network architecture:** two hidden layers with 32 neurons each and ELU activation

In [ ]:
def create_q_model(state_shape, action_size):

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation = 'elu', input_shape = state_shape),
        tf.keras.layers.Dense(32, activation = 'elu'),
        tf.keras.layers.Dense(action_size, activation = None)
    ])

    return model

state_shape = env.observation_space.shape
action_size = env.action_space.n
model = create_q_model(state_shape, action_size)
model.summary()

## 3. Create the Replay Buffer and Policy

In traditional Q-learning with a table, we update values immediately after every step. However, deep neural networks are sensitive to correlations. If we train the network on consecutive frames of the game (where the cart barely moves between frames), the network essentially "overfits" to the current situation and forgets what it learned earlier.

To fix this, we use Experience Replay. We save the agent's steps in a memory buffer and then randomly sample a batch of past experiences to train the network.

**Replay Buffer:** Stores `(state, action, reward, next_state, done)` tuples in a double-ended queue (deque).

**Sampling:** The sample() method grabs a random batch, breaking that sequential correlation.

**Epsilon-Greedy Policy:** With probability `epsilon`, it picks a random move; otherwise, it asks the neural network for the best move


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity = 10000):
        self.buffer = deque(maxlen=capacity)

    def put(self, state, action, reward, next_state, done):
        self.buffer.append((state,action,reward,next_state,done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size) # we pick random samples from the buffer to mitigate
                                                      #correlation issues and catastrophic forgetting

    def size(self):
        return len(self.buffer)

def epsilon_greedy_policy(state, model, epsilon=0.1):
    if np.random.rand() < epsilon:
        return np.random.randint(action_size) # Exploration
    else:
        q_values = model(np.array([state]))
        return np.argmax(q_values[0]) # Exploitation

## Hyperparameters

In [ ]:
batch_size = 32
gamma = 0.95
optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-3)
loss_fn = tf.keras.losses.MeanSquaredError()

## 4. Vanilla DQN (Moving Target)

* This is the most basic form of Q-learning
* A single neural network is used to both predict the Q-value of the current state and estimate the target Q-value for the next state.
* Because the same network calculates the prediction and the target, every time you update the weights to move the prediction closer to the target, the target also moves!
* This feedback loop causes the network to diverge.
* Generally considered unstable.

In [ ]:

model = create_q_model(state_shape, action_size)

# The training step
@tf.function # Compiles the function for faster execution

def train_step(states, actions, rewards, next_states, dones, model):

    # calculate the target Q-values (ground truth)
    # Formula: Q_target = r + gamma * max_a' Q(next_state, a')
    next_q_values = model(next_states)
    max_next_q = tf.reduce_max(next_q_values, axis=1)
    target_q = rewards + (1 - dones)*gamma*max_next_q

    with tf.GradientTape() as tape:
        # calculate predicted Q-values

        q_values = model(states)

        #we only want the Q-values for the actions taken
        # this creates a mask to select those specific Q-values
        masks = tf.one_hot(actions,2) # 2 actions in CartPole
        predicted_q = tf.reduce_sum(q_values * masks, axis=1)

        # calculate loss
        loss = loss_fn(target_q, predicted_q)

    # backpropagation
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# Main Training Loop
memory = ReplayBuffer(10000)
rewards_history = []
episodes = 300

print("Starting training...")
for episode in range(episodes):
    state, _ = env.reset()
    total_rewards = 0
    done  = False

    while not done:
        # 1. Action Selection ( Epsilon decay)
        epsilon = max(0.01, 1.0 - episode / (episodes / 2)) # Decay exploration over time
        action = epsilon_greedy_policy(state, model, epsilon)

        # 2. Step the environment
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3. Store experience
        memory.put(state, action, reward, next_state, done)
        state = next_state
        total_rewards += reward

        # 4. Train (only if we have enough memories)
        if memory.size() > batch_size:
            batch = memory.sample(batch_size)
            # Unpack batch into numpy arrays
            states, actions, rewards_b, next_states, dones = zip(*batch)
            train_step(np.array(states), np.array(actions),
                       np.array(rewards_b, dtype=np.float32),
                       np.array(next_states), np.array(dones, dtype=np.float32), model)

    rewards_history.append(total_rewards)
    if episode % 50 == 0:
        print(f"Episode {episode}: Total Reward = {total_rewards}")

# Plot the learning progress
plt.plot(rewards_history)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.show()




## DQN with Fixed Q-Targets

This is the standard Vanilla DQN introduced by DeepMind in 2013 that made the breakthrough in Atari games.

* Used two separate networks: and Online model (learns every step) and a Target Model (frozen clone).
* The target model is used exclusively to calculate the target values.
* The weights of the Online Model are copied to the Target model only periodically.
* **Advantage - Stability:** The target remains stationary for a while, giving the online model a stable goal to chase.
* **Disadvantage - Overestimation:** It tends to overestimate Q-values because the `max` operator picks the highest value from the target network, essentially treating noise as a signal.
* **Improvement over Naive DQN:** Solves the divergence issue caused by the moving target.


## Double DQN

* Introduced by DeepMind in 2015 to address the overestimation issue in the previous algorithm.

* It decouples selection from evaluation. It forces the two networks to cooperate.

* **Selection:** We ask the Online Model (the one currently learning) to choose the best action for the next state.

* **Evaluation:** We ask the Target Model to calculate the value of that specific action.

* **Advantage - Accuracy:** Reduces overestimation of Q-values, leading to more reliable learning.

* **Improvement over Fixed Target DQN:** Fixes the positive bias (overestimation) of Q-values that occurs when using `max` on noisy estimates.
  

# Dueling DQN

This introduces an architectural change to the Neural Network itself, rather than the training rule.

* The network splits into two sepratae streams of layers after the initial feature extraction:
    * **Value Stream V(s):** Estimates how good the current state is overall.
    * **Advantage Stream A(s,a):** Estimates how much better one action is compared to others.

    * These are aggregated at the end to produce the final Q-value:

  $Q(s,a) = V(s) + A(s,a)$

* **Advantage:** This can learn the value of a state without having to learn the value of every single action at that state. This is crucial for states where the choice of action doesn't matter.

* **Improvement over Double DQN:** Faster convergence and better sample efficiency, particularly in environments with many states where the action choice has little impact on the immediate outcome.


